# Multivariate Exploratory Data Analysis

This notebook continues from the bivariate EDA and explores **feature-to-feature relationships** and validates modeling assumptions.

## Objectives
1. Analyze correlations and multicollinearity among numeric features
2. Measure categorical feature associations (Cramér's V)
3. Investigate feature interaction effects
4. Assess non-linear feature relevance via mutual information
5. Validate model assumptions (class balance, separability, missing values)

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import ttest_ind
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

In [ ]:
df = pd.read_parquet("../data/silver/data_to_bivar.parquet", engine='pyarrow')
df = df.sort_values(["date", "time"]).reset_index(drop=True)
df.head()

## 1. Multivariate Analysis & Pre-Modeling Checks

The bivariate EDA (notebook 05) analyzed individual features against the target. Before modeling we need to understand **how features relate to each other** and verify that the data satisfies the assumptions of the models we plan to use (Random Forest, XGBoost, LightGBM).

### What we need to check:
1. **Correlation heatmap** — pairwise linear relationships among numeric features
2. **Multicollinearity (VIF)** — redundant information that inflates variance in linear models and can still affect tree splits
3. **Cramér's V** — association strength between categorical features
4. **Feature interaction effects** — does the combination of two features reveal patterns invisible in isolation?
5. **Model assumption validation** — class separability, feature scale review, missing-value summary

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import LabelEncoder

# Impute VTAT missing values with median for correlation analysis
df['avg_vtat_imputed'] = df['avg_vtat'].fillna(df['avg_vtat'].median())

# Create binary and cyclical features
df['is_morning_rush'] = ((df['hour'] >= 7) & (df['hour'] <= 10)).astype(int)
df['is_evening_rush'] = ((df['hour'] >= 17) & (df['hour'] <= 21)).astype(int)
df['is_peak_hour'] = (df['is_morning_rush'] | df['is_evening_rush']).astype(int)
df['is_late_night'] = ((df['hour'] >= 23) | (df['hour'] <= 5)).astype(int)
df['is_high_vtat'] = (df['avg_vtat_imputed'] >= 15).astype(int)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

NUMERIC_FEATURES = [
    'avg_vtat_imputed', 'is_high_vtat',
    'hour', 'dayofweek', 'month',
    'is_weekend', 'is_peak_hour', 'is_late_night',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'is_cancelled'
]

numeric_df = df[NUMERIC_FEATURES].copy()
print(f"Numeric feature matrix shape: {numeric_df.shape}")
numeric_df.head()

### 1.1 Correlation Heatmap

Pearson correlation between all numeric features. Values close to ±1 indicate strong linear relationships; values near 0 indicate no linear relationship.

For tree-based models correlated features won't break the model, but they dilute feature importance and make interpretation harder.

In [ ]:
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(
    corr_matrix, mask=mask, cmap=cmap, center=0,
    annot=True, fmt='.2f', square=True, linewidths=0.5,
    vmin=-1, vmax=1, ax=ax,
    cbar_kws={"shrink": 0.8, "label": "Pearson Correlation"}
)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Highlight strong correlations (|r| > 0.5) excluding self-correlations
print("\nFeature pairs with |correlation| > 0.5:")
print("=" * 60)
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {corr_matrix.columns[i]:25s} vs {corr_matrix.columns[j]:25s}  r = {r:+.3f}")

### 1.2 Variance Inflation Factor (VIF)

VIF measures how much a feature's variance is inflated by multicollinearity with other features. 

- **VIF = 1**: No multicollinearity
- **VIF = 1–5**: Moderate, generally acceptable
- **VIF = 5–10**: High multicollinearity, may need attention
- **VIF > 10**: Severe multicollinearity, consider removing or combining features

While Random Forest and XGBoost are robust to multicollinearity, checking VIF helps us understand which features carry redundant information and ensures that feature importance rankings are interpretable.

In [ ]:
# VIF on features only (exclude target)
feature_cols_for_vif = [c for c in NUMERIC_FEATURES if c != 'is_cancelled']
vif_data = numeric_df[feature_cols_for_vif].dropna()

vif_results = pd.DataFrame({
    'Feature': feature_cols_for_vif,
    'VIF': [variance_inflation_factor(vif_data.values, i) for i in range(len(feature_cols_for_vif))]
}).sort_values('VIF', ascending=False)

print("VARIANCE INFLATION FACTORS")
print("=" * 50)
for _, row in vif_results.iterrows():
    flag = "⚠️ HIGH" if row['VIF'] > 5 else ("⚡ MODERATE" if row['VIF'] > 2 else "✅ OK")
    print(f"  {row['Feature']:25s}  VIF = {row['VIF']:8.2f}  {flag}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['red' if v > 5 else ('orange' if v > 2 else 'green') for v in vif_results['VIF']]
ax.barh(vif_results['Feature'], vif_results['VIF'], color=colors, edgecolor='black')
ax.axvline(x=5, color='red', linestyle='--', linewidth=1.5, label='High threshold (5)')
ax.axvline(x=2, color='orange', linestyle='--', linewidth=1.5, label='Moderate threshold (2)')
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factor per Feature')
ax.legend()
plt.tight_layout()
plt.show()

### 1.3 Cramér's V — Categorical Feature Associations

Cramér's V quantifies the strength of association between two categorical variables (values range from 0 to 1):

- **0**: No association
- **0–0.1**: Negligible
- **0.1–0.3**: Weak
- **0.3–0.5**: Moderate
- **> 0.5**: Strong

This helps us understand whether categorical features (vehicle_type, locations, temporal categories) carry independent information or are redundant.

In [ ]:
def cramers_v(x, y):
    """Compute Cramér's V statistic for two categorical Series."""
    confusion = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion)[0]
    n = confusion.sum().sum()
    min_dim = min(confusion.shape) - 1
    if min_dim == 0:
        return 0
    return np.sqrt(chi2 / (n * min_dim))

# Categorical columns to analyze
cat_cols = ['vehicle_type', 'vtat_bucket', 'is_weekend', 'is_peak_hour', 'is_late_night', 'is_high_vtat', 'is_cancelled']

# Compute pairwise Cramér's V
cramers_matrix = pd.DataFrame(
    np.zeros((len(cat_cols), len(cat_cols))),
    index=cat_cols, columns=cat_cols
)

for i, col_i in enumerate(cat_cols):
    for j, col_j in enumerate(cat_cols):
        if i == j:
            cramers_matrix.loc[col_i, col_j] = 1.0
        elif i < j:
            v = cramers_v(df[col_i].astype(str), df[col_j].astype(str))
            cramers_matrix.loc[col_i, col_j] = v
            cramers_matrix.loc[col_j, col_i] = v

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(cramers_matrix, dtype=bool))
sns.heatmap(
    cramers_matrix.astype(float), mask=mask, annot=True, fmt='.3f',
    cmap='YlOrRd', vmin=0, vmax=1, square=True, linewidths=0.5, ax=ax,
    cbar_kws={"shrink": 0.8, "label": "Cramér's V"}
)
ax.set_title("Cramér's V — Categorical Feature Associations", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Highlight noteworthy pairs
print("\nCramér's V interpretation (excluding self-pairs):")
print("=" * 60)
for i in range(len(cat_cols)):
    for j in range(i + 1, len(cat_cols)):
        v = cramers_matrix.iloc[i, j]
        strength = "STRONG" if v > 0.5 else ("MODERATE" if v > 0.3 else ("WEAK" if v > 0.1 else "NEGLIGIBLE"))
        if v > 0.1:
            print(f"  {cat_cols[i]:20s} vs {cat_cols[j]:20s}  V = {v:.3f}  ({strength})")

### 1.4 Feature Interaction Effects

Bivariate analysis showed that VTAT is the dominant predictor. But does VTAT interact with other features? For example:
- Does high VTAT cause more cancellations for certain vehicle types?
- Does the effect of VTAT change during peak hours vs. off-peak?
- Is there a location × VTAT interaction?

Understanding interactions helps us decide whether to create interaction features and explains why ensemble methods like RF/XGBoost may outperform linear models.

In [ ]:
# --- Interaction 1: VTAT bucket × Vehicle Type ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cancellation rate: vtat_bucket × vehicle_type
interaction_1 = df.groupby(['vtat_bucket', 'vehicle_type'], observed=True)['is_cancelled'].mean().unstack()
interaction_1.plot(kind='bar', ax=axes[0], width=0.8, edgecolor='black')
axes[0].set_title('Cancellation Rate: VTAT Bucket × Vehicle Type')
axes[0].set_xlabel('VTAT Bucket')
axes[0].set_ylabel('Cancellation Rate')
axes[0].legend(title='Vehicle Type', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[0].set_ylim(0, 1.05)

# Cancellation rate: vtat_bucket × peak_hour
interaction_2 = df.groupby(['vtat_bucket', 'is_peak_hour'], observed=True)['is_cancelled'].mean().unstack()
interaction_2.columns = ['Off-Peak', 'Peak Hour']
interaction_2.plot(kind='bar', ax=axes[1], width=0.6, color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_title('Cancellation Rate: VTAT Bucket × Peak Hour')
axes[1].set_xlabel('VTAT Bucket')
axes[1].set_ylabel('Cancellation Rate')
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("-" * 60)
for bucket in interaction_1.index:
    rates = interaction_1.loc[bucket]
    spread = rates.max() - rates.min()
    print(f"  VTAT {bucket}: Vehicle type spread = {spread:.3f} (max={rates.idxmax()}: {rates.max():.2%}, min={rates.idxmin()}: {rates.min():.2%})")

In [ ]:
# --- Interaction 2: VTAT × Weekend/Weekday ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# VTAT distribution by cancellation status & weekend
for i, (weekend_val, label) in enumerate([(0, 'Weekday'), (1, 'Weekend')]):
    subset = df[df['is_weekend'] == weekend_val]
    cancelled = subset[subset['is_cancelled'] == 1]['avg_vtat_imputed']
    completed = subset[subset['is_cancelled'] == 0]['avg_vtat_imputed']
    
    axes[i].hist(completed, bins=30, alpha=0.6, label='Completed', color='green', density=True, edgecolor='black')
    axes[i].hist(cancelled, bins=30, alpha=0.6, label='Cancelled', color='red', density=True, edgecolor='black')
    axes[i].set_title(f'VTAT Distribution — {label}')
    axes[i].set_xlabel('avg_vtat_imputed')
    axes[i].set_ylabel('Density')
    axes[i].legend()

plt.tight_layout()
plt.show()

# Statistical comparison: VTAT effect on weekday vs weekend
for weekend_val, label in [(0, 'Weekday'), (1, 'Weekend')]:
    subset = df[df['is_weekend'] == weekend_val]
    corr, pval = stats.pointbiserialr(subset['is_cancelled'], subset['avg_vtat_imputed'])
    print(f"  {label}: Point-biserial r = {corr:.4f}, p = {pval:.2e}")

In [ ]:
# --- Interaction 3: Hour × Vehicle Type Cancellation Heatmap ---
hourly_vehicle = df.groupby(['hour', 'vehicle_type'], observed=True)['is_cancelled'].mean().unstack()

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    hourly_vehicle.T, cmap='RdYlGn_r', annot=True, fmt='.2f',
    linewidths=0.5, ax=ax, vmin=0.20, vmax=0.45,
    cbar_kws={"label": "Cancellation Rate"}
)
ax.set_title('Cancellation Rate Heatmap: Hour × Vehicle Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Vehicle Type')
plt.tight_layout()
plt.show()

### 1.5 Mutual Information — Non-Linear Feature Relevance

Mutual Information (MI) captures both linear and non-linear dependencies between each feature and the target. Unlike Pearson correlation, MI can detect any type of statistical dependence. Higher MI means the feature carries more information about the target.

This is particularly relevant for tree-based models which can exploit non-linear patterns.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

feature_cols = [c for c in NUMERIC_FEATURES if c != 'is_cancelled']
X_mi = numeric_df[feature_cols].dropna()
y_mi = numeric_df.loc[X_mi.index, 'is_cancelled']

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42, n_neighbors=5)
mi_df = pd.DataFrame({
    'Feature': feature_cols,
    'MI Score': mi_scores
}).sort_values('MI Score', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(mi_df)))
ax.barh(mi_df['Feature'], mi_df['MI Score'], color=colors, edgecolor='black')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Mutual Information: Feature Relevance for Cancellation')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nMutual Information Ranking:")
print("=" * 50)
for _, row in mi_df.iterrows():
    bar = "█" * int(row['MI Score'] * 200)
    print(f"  {row['Feature']:25s}  MI = {row['MI Score']:.4f}  {bar}")

### 1.6 Model Assumption Validation

We are planning to use **Random Forest**, **XGBoost** and **LightGBM** — all tree-based ensemble methods. Let's check their key assumptions:

| Assumption | RF/XGBoost/LightGBM | Check |
|------------|---------------------|-------|
| Linearity | NOT required | Tree-based models handle non-linear relationships natively |
| Normality | NOT required | No distributional assumptions on features |
| Homoscedasticity | NOT required | Variance equality is irrelevant for tree splits |
| Feature scale | NOT sensitive | Trees split on rank-order, not magnitude |
| Multicollinearity | Robust but affects interpretability | Checked above via VIF |
| Class balance | Sensitive | Need to verify and handle via class_weight or sampling |
| Sufficient data | Required | Need enough samples per class |
| Feature independence | NOT required | Trees naturally capture interactions |

**Key checks below:** class separability, class balance, and whether features have enough variance.

In [ ]:
# --- Class Balance ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = df['is_cancelled'].value_counts()
class_labels = ['Completed (0)', 'Cancelled (1)']

axes[0].bar(class_labels, class_counts.values, color=['green', 'red'], edgecolor='black')
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution')
for i, (label, count) in enumerate(zip(class_labels, class_counts.values)):
    axes[0].text(i, count + 500, f'{count:,}\n({count/len(df):.1%})', ha='center', fontsize=11)

# Imbalance ratio
majority = class_counts.max()
minority = class_counts.min()
ratio = majority / minority

axes[1].barh(['Imbalance Ratio'], [ratio], color='orange', edgecolor='black')
axes[1].set_xlabel('Majority / Minority')
axes[1].set_title(f'Imbalance Ratio: {ratio:.2f}:1')
axes[1].axvline(x=1.5, color='green', linestyle='--', label='Slight (1.5)')
axes[1].axvline(x=3.0, color='red', linestyle='--', label='Moderate (3.0)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nClass balance summary:")
print(f"  Completed: {class_counts[0.0]:,} ({class_counts[0.0]/len(df):.1%})")
print(f"  Cancelled: {class_counts[1.0]:,} ({class_counts[1.0]/len(df):.1%})")
print(f"  Ratio:     {ratio:.2f}:1")
print(f"\n  → Moderate imbalance. Will use class_weight='balanced' and threshold tuning.")

In [ ]:
# --- Class Separability: VTAT distribution by class ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# KDE of VTAT by class
for cls, color, label in [(0, 'green', 'Completed'), (1, 'red', 'Cancelled')]:
    subset = df[df['is_cancelled'] == cls]['avg_vtat_imputed']
    axes[0].hist(subset, bins=50, alpha=0.5, density=True, color=color, label=label, edgecolor='black')
axes[0].set_title('VTAT Distribution by Class (Strongest Predictor)')
axes[0].set_xlabel('avg_vtat_imputed')
axes[0].set_ylabel('Density')
axes[0].legend()

# Feature variance check
feature_vars = numeric_df[feature_cols].var().sort_values(ascending=False)
axes[1].barh(feature_vars.index, feature_vars.values, color='steelblue', edgecolor='black')
axes[1].set_xlabel('Variance')
axes[1].set_title('Feature Variance (higher = more informative splits)')
axes[1].axvline(x=0.01, color='red', linestyle='--', label='Near-zero variance threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Identify near-zero variance features
low_var = feature_vars[feature_vars < 0.01]
if len(low_var) > 0:
    print(f"\n⚠️ Features with near-zero variance (< 0.01):")
    for feat, var in low_var.items():
        print(f"    {feat}: variance = {var:.6f}")
else:
    print("\n✅ All features have sufficient variance for tree splits.")

In [ ]:
# --- Missing value summary for modeling readiness ---
print("MISSING VALUE SUMMARY — Full Dataset")
print("=" * 60)
missing = df[NUMERIC_FEATURES + ['vehicle_type', 'pickup_location', 'drop_location']].isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing_summary) > 0:
    print(missing_summary)
    print(f"\n  → avg_vtat has {missing_summary.loc['avg_vtat_imputed', 'Missing %'] if 'avg_vtat_imputed' in missing_summary.index else 0}% missing")
    print("  → Will impute using median by vehicle_type (preserves within-group distribution)")
else:
    print("  ✅ No missing values in the feature matrix after imputation.")

## 2. Multivariate EDA Summary & Implications for Modeling

### Key Findings

**Multivariate Analysis (Feature ↔ Feature):**
| Check | Result | Action |
|-------|--------|--------|
| Correlation heatmap | Expected correlations between raw & cyclical encodings of the same variable | Keep both: raw for trees, cyclical for potential linear baselines |
| VIF | `hour`/`hour_sin`/`hour_cos` and similar pairs may show moderate VIF | Acceptable for tree-based models; monitor feature importance |
| Cramér's V | `vtat_bucket` and `is_high_vtat` are related (by construction) | Expected — both derive from VTAT |
| Feature interactions | VTAT effect is consistent across vehicle types and time periods | No strong interaction terms needed beyond VTAT itself |
| Mutual Information | VTAT dominates; temporal and location features carry weak signal | Confirms bivariate findings with non-linear measure |

**Model Assumption Checks:**
| Assumption | Status | Notes |
|------------|--------|-------|
| Class balance | ⚠️ Moderate imbalance (~2:1) | Use `class_weight='balanced'` + threshold tuning |
| Class separability | ✅ Clear separation via VTAT | VTAT > 15 = near-certain cancellation |
| Feature variance | ✅ Sufficient for all features | No near-zero variance issues |
| Missing values | ⚠️ `avg_vtat` has 7% missing | Impute with median by vehicle_type |

### Implications for Feature Engineering
1. **VTAT is the star feature** — create `avg_vtat_imputed`, `vtat_bucket`, `is_high_vtat`
2. **Location needs encoding** — 176 unique values → target encoding with smoothing
3. **Vehicle type** is weak but cheap to include → label encoding
4. **Temporal features** are weak individually but may help in ensemble splits → include cyclical + binary encodings
5. **No strong interactions** justify complex interaction features for v1

### Implications for Model Selection
- **Tree-based models are ideal**: no normality/linearity assumptions, robust to multicollinearity, handle mixed feature types, can capture non-linear VTAT threshold effect
- **Class imbalance handling** is critical: use balanced class weights + F2-optimized threshold
- **Feature importance will be dominated by VTAT** — this is expected and validated by both correlation and mutual information analysis